In [1]:
#Cell1
import os, random, warnings, torch, math
import numpy as np
import pandas as pd
import rasterio
import cv2
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
KAGGLE_BASE = '/kaggle/input/datasets/jucor1/worldstrat'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
import os, random, warnings, torch, pandas as pd, rasterio, cv2, numpy as np
from torch.utils.data import Dataset, DataLoader

def fast_find_metadata():
    likely_paths = [
        '/kaggle/input/datasets/jucor1/worldstrat/metadata.csv',
        '/kaggle/input/worldstrat/metadata.csv',
        '/kaggle/input/worldstrat-dataset/metadata.csv'
    ]
    for p in likely_paths:
        if os.path.exists(p): return p
    return None

META_FILE_PATH = fast_find_metadata()
KAGGLE_BASE = os.path.dirname(META_FILE_PATH) if META_FILE_PATH else '/kaggle/input/worldstrat' 

class WorldStratSR(Dataset):
    def __init__(self, base_dir=KAGGLE_BASE, land_class=None, patch_size=128, max_samples=3000, seed=42, is_val=False):
        self.patch_size = patch_size
        self.lr_dir = os.path.join(base_dir, 'lr_dataset')
        self.hr_dir = os.path.join(base_dir, 'hr_dataset', '12bit')
        
        df = pd.read_csv(META_FILE_PATH)
        aoi_col = 'Unnamed: 0' if 'Unnamed: 0' in df.columns else df.columns[0]
        valid_aois = df[aoi_col].unique().tolist()
        
        self.pairs = []
        random.seed(seed)
        random.shuffle(valid_aois)
        
        for aoi in valid_aois:
            if len(self.pairs) >= max_samples: break
            hr_path = os.path.join(self.hr_dir, aoi, f"{aoi}_rgbn.tiff")
            lr_l2a_dir = os.path.join(self.lr_dir, aoi, 'L2A')
            
            if os.path.exists(hr_path) and os.path.exists(lr_l2a_dir):
                all_revs = [os.path.join(lr_l2a_dir, f) for f in os.listdir(lr_l2a_dir) if 'L2A_data' in f]
                if not all_revs: continue
                
                best_lr, best_score = None, -1.0
                for rev in all_revs[:6]:
                    try:
                        with rasterio.open(rev) as src:
                            rgb = src.read([4, 3, 2], out_shape=(3, 32, 32)).astype(np.float32)
                            if rgb.max() > 10.0: rgb /= 10000.0
                            q90_min = np.percentile(rgb.min(axis=0), 90)
                            q90_diff = np.abs(np.percentile(rgb[0], 90) - np.percentile(rgb[2], 90))
                            if q90_min > 0.20 and q90_diff < 0.05: continue
                            score = rgb.std()
                            if score > best_score: best_score, best_lr = score, rev
                    except: continue
                
                if best_lr: self.pairs.append((best_lr, hr_path))

        n_val = max(10, len(self.pairs) // 10)
        self.pairs = self.pairs[:n_val] if is_val else self.pairs[n_val:]

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        lr_path, hr_path = self.pairs[idx]
        with rasterio.open(hr_path) as src:
            hr = src.read([1, 2, 3]).astype(np.float32)
            if hr.max() > 10.0: hr /= 4095.0
        with rasterio.open(lr_path) as src:
            if src.count >= 4: lr = src.read([4, 3, 2]).astype(np.float32)
            elif src.count >= 3: lr = src.read([1, 2, 3]).astype(np.float32)
            else:
                one = src.read(1).astype(np.float32)
                lr = np.stack([one, one, one])
            if lr.max() > 10.0: lr /= 10000.0
        hr, lr = np.clip(hr, 0, 1), np.clip(lr, 0, 1)
        hr_h, hr_w = hr.shape[1], hr.shape[2]
        lr_up = np.stack([cv2.resize(lr[c], (hr_w, hr_h), interpolation=cv2.INTER_CUBIC) for c in range(3)])
        ps = self.patch_size
        y, x = (hr_h - ps)//2, (hr_w - ps)//2
        return torch.from_numpy(lr_up[:, y:y+ps, x:x+ps]), torch.from_numpy(hr[:, y:y+ps, x:x+ps])

val_ds = WorldStratSR(max_samples=3000, is_val=True)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False)
train_ds = WorldStratSR(max_samples=3000, is_val=False)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)

🔍 Building ELITE dataset (3,000 samples)...
   ... Clean pairs found: 250 / 3000
   ... Clean pairs found: 500 / 3000
   ... Clean pairs found: 750 / 3000
   ... Clean pairs found: 1000 / 3000
   ... Clean pairs found: 1250 / 3000
   ... Clean pairs found: 1500 / 3000
   ... Clean pairs found: 1750 / 3000
   ... Clean pairs found: 2000 / 3000
   ... Clean pairs found: 2250 / 3000


In [ ]:
audit_loader = DataLoader(val_ds, batch_size=20, shuffle=True)
lr_audit, _ = next(iter(audit_loader))
fig, axes = plt.subplots(4, 5, figsize=(20, 16))
for i in range(20):
    img = lr_audit[i].numpy().transpose(1, 2, 0)
    mi, ma = np.percentile(img, (2, 98))
    img = np.clip((img - mi) / (ma - mi + 1e-8), 0, 1)
    axes[i // 5, i % 5].imshow(img)
    axes[i // 5, i % 5].axis('off')
plt.show()

In [ ]:
class STN(nn.Module):
    def __init__(self):
        super().__init__()
        self.loc = nn.Sequential(
            nn.Conv2d(3, 32, 7, padding=3), nn.MaxPool2d(2), nn.ReLU(True),
            nn.Conv2d(32, 64, 5, padding=2), nn.MaxPool2d(2), nn.ReLU(True),
            nn.AdaptiveAvgPool2d(1)
        )
        self.fc = nn.Sequential(nn.Linear(64, 32), nn.ReLU(True), nn.Linear(32, 6))
        self.fc[2].weight.data.zero_()
        self.fc[2].bias.data.copy_(torch.tensor([1, 0, 0, 0, 1, 0], dtype=torch.float))

    def forward(self, x):
        theta = self.fc(self.loc(x).view(-1, 64)).view(-1, 2, 3)
        grid = F.affine_grid(theta, x.size(), align_corners=False)
        return F.grid_sample(x, grid, align_corners=False)

class RCAB(nn.Module):
    def __init__(self, n_feats=64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(n_feats, n_feats, 3, padding=1), 
            nn.ReLU(True), 
            nn.Conv2d(n_feats, n_feats, 3, padding=1)
        )
        self.ca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), 
            nn.Conv2d(n_feats, 4, 1), 
            nn.ReLU(True), 
            nn.Conv2d(4, n_feats, 1), 
            nn.Sigmoid()
        )

    def forward(self, x):
        res = self.conv(x)
        return x + (res * self.ca(res)) * 0.1

class SatelliteSR(nn.Module):
    def __init__(self, n_blocks=16, n_feats=64):
        super().__init__()
        self.stn = STN()
        self.head = nn.Conv2d(3, n_feats, 3, padding=1)
        self.body = nn.Sequential(*[RCAB(n_feats) for _ in range(n_blocks)])
        self.tail = nn.Conv2d(n_feats, 3, 3, padding=1)

    def forward(self, x):
        x = self.stn(x)
        return x + self.tail(self.body(self.head(x)))

model = SatelliteSR().to(device)

In [ ]:

class CombinedLoss(nn.Module):
    def __init__(self, device):
        super().__init__()
        vgg = models.vgg19(weights='IMAGENET1K_V1').features[:18].eval().to(device)
        for p in vgg.parameters(): p.requires_grad = False
        self.vgg, self.l1 = vgg, nn.L1Loss()
    def forward(self, sr, hr):
        l1 = self.l1(sr, hr)
        perc = F.l1_loss(self.vgg(sr), self.vgg(hr))
        return l1 + 0.5 * perc, l1, perc

criterion = CombinedLoss(device)

In [ ]:
import torchvision.models as models

In [ ]:
import torch.optim as optim
optimizer = optim.AdamW(model.parameters(), lr=2e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200)

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SatelliteSR().to(device)

model_path = "/kaggle/input/models/hassanali5744/11/pytorch/default/1/high_res_stn_model5.pth"
checkpoint = torch.load(model_path, map_location=device)

model.load_state_dict(checkpoint, strict=True)
model.eval()

print("✅ Model loaded successfully!")

In [ ]:
START_EPOCH = 1
END_EPOCH = 200

for epoch in range(START_EPOCH, END_EPOCH):
    model.train()
    loss_total = 0
    
    for lr_imgs, hr_imgs in train_loader:
        lr_imgs, hr_imgs = lr_imgs.to(device), hr_imgs.to(device)
        
        optimizer.zero_grad()
        outputs = model(lr_imgs)
        loss, _, _ = criterion(outputs, hr_imgs)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()
        
        loss_total += loss.item()
    
    scheduler.step()
    avg_loss = loss_total / len(train_loader)
    print(f"Epoch [{epoch+1}/{END_EPOCH}] Loss: {avg_loss:.5f}")

    if (epoch + 1) % 20 == 0:
        torch.save(model.state_dict(), 'high_res_stn_model.pth')
        print(f"✨ Progress Saved: Epoch {epoch+1}")

torch.save(model.state_dict(), 'high_res_stn_model.pth')
print("🏁 Training complete.")

In [ ]:
import torch
import numpy as np 
import matplotlib.pyplot as plt
import random

model.eval()

val_iter = iter(val_loader)
rand_batch_idx = random.randint(0, len(val_loader) - 1)

for _ in range(rand_batch_idx + 1):
    lr_b, hr_b = next(val_iter)

with torch.no_grad():
    sr_b = model(lr_b.to(device))

batch_size = min(4, lr_b.size(0))
fig, axes = plt.subplots(batch_size, 3, figsize=(15, 5 * batch_size))
titles = ["LR", "SR", "HR"]

for i in range(batch_size):
    imgs = [lr_b[i], sr_b[i], hr_b[i]]
    for j, (img, title) in enumerate(zip(imgs, titles)):
        img_np = img.cpu().numpy().transpose(1, 2, 0)

        mi, ma = np.percentile(img_np, (2, 98))
        img_np = np.clip((img_np - mi) / (ma - mi + 1e-8), 0, 1)

        ax = axes[i, j] if batch_size > 1 else axes[j]
        ax.imshow(img_np)
        ax.set_title(title)
        ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import mlflow
import mlflow.pytorch
import lpips
import numpy as np
import math
import torch.nn.functional as F

loss_fn_lpips = lpips.LPIPS(net='vgg').to(device)

def calc_psnr(sr, hr):
    mse = torch.mean((sr - hr) ** 2).item()
    return 100 if mse == 0 else 20 * math.log10(1.0 / math.sqrt(mse))

def calc_ssim(sr, hr):
    C1, C2 = 0.01**2, 0.03**2
    mu_x = F.avg_pool2d(sr, 3, stride=1, padding=1)
    mu_y = F.avg_pool2d(hr, 3, stride=1, padding=1)
    sigma_x_sq = F.avg_pool2d(sr**2, 3, stride=1, padding=1) - mu_x**2
    sigma_y_sq = F.avg_pool2d(hr**2, 3, stride=1, padding=1) - mu_y**2
    sigma_xy = F.avg_pool2d(sr*hr, 3, stride=1, padding=1) - mu_x*mu_y
    ssim = ((2*mu_x*mu_y + C1)*(2*sigma_xy + C2)) / ((mu_x**2 + mu_y**2 + C1)*(sigma_x_sq + sigma_y_sq + C2))
    return ssim.mean().item()

mlflow.set_experiment("WorldStrat_STN_Final")

with mlflow.start_run(run_name="STN_Alignment_Success"):
    model.eval()
    psnr_vals, ssim_vals, lpips_vals = [], [], []
    
    print("📊 Evaluating performance...")
    with torch.no_grad():
        for i, (lr_v, hr_v) in enumerate(val_loader):
            lr_v, hr_v = lr_v.to(device), hr_v.to(device)
            sr_v = torch.clamp(model(lr_v), 0, 1)
            
            psnr_vals.append(calc_psnr(sr_v, hr_v))
            ssim_vals.append(calc_ssim(sr_v, hr_v))
            lpips_vals.append(loss_fn_lpips(sr_v*2-1, hr_v*2-1).mean().item())
            
            if i >= 10: break

    metrics = {
        "psnr": np.mean(psnr_vals),
        "ssim": np.mean(ssim_vals),
        "lpips": np.mean(lpips_vals)
    }

    for name, val in metrics.items():
        mlflow.log_metric(name, val)
    
    mlflow.pytorch.log_model(model, "final_stn_model")

    print("\n" + "="*30)
    print(f"🏆 FINAL PSNR:  {metrics['psnr']:.2f} dB")
    print(f"🏆 FINAL SSIM:  {metrics['ssim']:.4f}")
    print(f"🏆 FINAL LPIPS: {metrics['lpips']:.4f}")
    print("="*30)

torch.save(model.state_dict(), 'high_res_stn_model5.pth')

In [ ]:
pip install mlflow lpips torchmetrics torchvision

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [5]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cuda
